# Notebook 03b — Fix BSISO Amplitude Data Bug
**Project:** ENSO-BSISO Self-Supervised Learning  
**Author:** Jiayi (jh9141@nyu.edu)

## Why this notebook exists

Notebook 07c (Option B) ran a `radius vs BSISO amplitude` diagnostic and got contradictory numbers:
- Pearson r = 0.023 (near zero)
- Spearman r = 0.747 (very strong)

The `radius_diagnostics.png` scatter showed why: the amplitude axis ran to **−1000**. The `bsiso_amplitude` column in `labels_aligned_mjjas_lee.csv` contains at least one −1000 fill value (the standard APEC missing-data sentinel) that:
- Destroys Pearson statistics on amplitude.
- Doesn't affect rank-based statistics (Spearman) because it's just one extreme rank.
- Doesn't affect training pipelines that filter on `amplitude > 1.0` (the bad value fails the filter).
- Doesn't affect probes on `bsiso_phase` or `enso_category` (different columns).

## What this notebook does

1. Loads `labels_aligned_mjjas_lee.csv`.
2. Identifies anomalous `bsiso_amplitude` values (negatives, fill values, gross outliers).
3. Reports counts and date ranges of bad values.
4. Saves a cleaned version `labels_aligned_mjjas_lee_clean.csv` with `NaN` in place of bad amplitudes (does NOT drop rows — row count and indices stay aligned with `X_MJJAS_lee.npy`).
5. Sanity-checks the clean version.

## Runtime

< 1 minute. Run on Colab once; notebooks 07c (rerunning diagnostic) and 08 (SSL temporal) can use the cleaned file.

---

## Cell 1 — Mount Drive + Load Labels

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_DIR    = '/content/drive/MyDrive/BSISO_SSL_Project'
PROCESSED_DIR  = f'{PROJECT_DIR}/data/processed'

LABELS_IN  = f'{PROCESSED_DIR}/labels_aligned_mjjas_lee.csv'
LABELS_OUT = f'{PROCESSED_DIR}/labels_aligned_mjjas_lee_clean.csv'

labels = pd.read_csv(LABELS_IN, parse_dates=['date'])
print(f'Loaded: {LABELS_IN}')
print(f'Rows:    {len(labels)}')
print(f'Columns: {list(labels.columns)}')
print(f'\nDate range: {labels["date"].min().date()} to {labels["date"].max().date()}')

## Cell 2 — Inspect `bsiso_amplitude` Distribution

A legitimate BSISO amplitude is a non-negative real number, typically 0 to ~3 (with rare excursions to ~5). Anything negative or above ~10 is suspect.

In [ ]:
amp = labels['bsiso_amplitude']

print('Summary statistics for bsiso_amplitude:')
print(amp.describe().to_string())
print(f'\nQuantiles (1st, 5th, 95th, 99th):')
print(f'  {amp.quantile([0.01, 0.05, 0.95, 0.99]).to_string()}')

# Count anomalies
n_neg       = int((amp < 0).sum())
n_neg_999   = int((amp <= -999).sum())          # standard fill values: -999, -9999, -1000
n_zero      = int((amp == 0).sum())
n_huge      = int((amp > 10).sum())
n_total     = len(amp)

print(f'\nAnomaly counts:')
print(f'  bsiso_amplitude < 0:       {n_neg}/{n_total}  ({n_neg/n_total*100:.2f}%)')
print(f'  bsiso_amplitude <= -999:   {n_neg_999}/{n_total}  ({n_neg_999/n_total*100:.2f}%)  ← fill values')
print(f'  bsiso_amplitude == 0:      {n_zero}/{n_total}  ({n_zero/n_total*100:.2f}%)')
print(f'  bsiso_amplitude > 10:      {n_huge}/{n_total}  ({n_huge/n_total*100:.2f}%)')

# List bad rows
bad_mask = (amp < 0) | (amp > 10)
if bad_mask.any():
    print(f'\nBad rows (n={int(bad_mask.sum())}):')
    print(labels.loc[bad_mask, ['date', 'bsiso_phase', 'bsiso_amplitude', 'enso_category']].to_string())
else:
    print('\nNo bad rows found.')

## Cell 3 — Histogram (Before Cleaning)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Full range — shows outliers
axes[0].hist(amp, bins=80, color='steelblue', alpha=0.7)
axes[0].set_xlabel('bsiso_amplitude'); axes[0].set_ylabel('Count')
axes[0].set_title('All values (full range — note any outliers)', fontweight='bold')
axes[0].grid(alpha=0.3)

# Plausible range only
axes[1].hist(amp[amp.between(0, 5)], bins=50, color='steelblue', alpha=0.7)
axes[1].set_xlabel('bsiso_amplitude (0–5 only)'); axes[1].set_ylabel('Count')
axes[1].set_title('Plausible range [0, 5]', fontweight='bold')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Cell 4 — Clean: Replace Anomalies with NaN, Save

Replace any `bsiso_amplitude < 0` or `bsiso_amplitude > 10` with `NaN`. Do NOT drop rows — keeping row count and indices aligned with `X_MJJAS_lee.npy` (6579 rows). Downstream code can drop NaN rows on the fly when computing amplitude-dependent statistics.

In [ ]:
labels_clean = labels.copy()
bad_mask = (labels_clean['bsiso_amplitude'] < 0) | (labels_clean['bsiso_amplitude'] > 10)
n_replaced = int(bad_mask.sum())
labels_clean.loc[bad_mask, 'bsiso_amplitude'] = np.nan

print(f'Replaced {n_replaced} bad bsiso_amplitude values with NaN.')
print(f'Row count unchanged: {len(labels_clean)} (must match X_MJJAS_lee.npy)')

# Re-summarise after cleaning
amp_clean = labels_clean['bsiso_amplitude']
print(f'\nAfter cleaning — bsiso_amplitude summary:')
print(amp_clean.describe().to_string())
print(f'  NaN count: {int(amp_clean.isna().sum())}')

# Save
labels_clean.to_csv(LABELS_OUT, index=False)
print(f'\nSaved: {LABELS_OUT}')
print(f'  Rows: {len(labels_clean)}  Columns: {list(labels_clean.columns)}')
print(f'\nDownstream notebooks (07c rerun, 08, 09) should load this cleaned file '
      f'instead of the original whenever amplitude is treated as a continuous variable.')

## Cell 5 — Verify: Histogram Comparison + Re-run Pearson Test

Quick re-test of `Pearson(amplitude, amplitude)` on the cleaned column to confirm it's now well-behaved. (Trivially r=1, but the goal is to verify there are no surprise NaN propagations.)

In [ ]:
from scipy.stats import pearsonr

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

bins = np.linspace(0, max(5, amp_clean.max()), 60)
axes[0].hist(amp.dropna(), bins=bins, color='red',  alpha=0.5, label=f'Before (n={len(amp.dropna())})')
axes[0].hist(amp_clean.dropna(), bins=bins, color='green', alpha=0.5, label=f'After  (n={int(amp_clean.notna().sum())})')
axes[0].set_xlabel('bsiso_amplitude (≥0 only)'); axes[0].set_ylabel('Count')
axes[0].set_title('Histogram — before (red) vs after cleaning (green)', fontweight='bold')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Sanity Pearson on cleaned column with itself (drops NaN automatically with .dropna())
clean = amp_clean.dropna().values
r, p = pearsonr(clean, clean)
print(f'Pearson(amplitude, amplitude) on cleaned data: r = {r:.4f}  (should be 1.0)')

# Anomalous-value mask info
axes[1].plot(labels['date'], labels['bsiso_amplitude'].clip(lower=-5, upper=5), '.', alpha=0.3, ms=2)
axes[1].plot(labels.loc[bad_mask, 'date'], [4.5]*int(bad_mask.sum()), 'rx', ms=10, label=f'replaced (n={int(bad_mask.sum())})')
axes[1].set_xlabel('Date'); axes[1].set_ylabel('bsiso_amplitude (clipped to [-5, 5])')
axes[1].set_title('Time series — replaced values marked with red ×', fontweight='bold')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f'\nDone. Clean file at: {LABELS_OUT}')

---
## Done!

**Output:** `data/processed/labels_aligned_mjjas_lee_clean.csv` — same row count as the original (6579), with `NaN` replacing anomalous `bsiso_amplitude` values.

**Usage in downstream notebooks:**
```python
labels = pd.read_csv(f'{PROCESSED_DIR}/labels_aligned_mjjas_lee_clean.csv', parse_dates=['date'])
# When computing amplitude-dependent statistics, use:
valid = labels['bsiso_amplitude'].notna()
# e.g., pearsonr(radii[valid], labels.loc[valid, 'bsiso_amplitude'])
```

---
*DDCS Project | jh9141@nyu.edu*